In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Estymacja energii stanu podstawowego łańcucha Heisenberga metodą VQE

*Szacowane zużycie zasobów: dwie minuty na procesorze Eagle r3 (UWAGA: To tylko szacunek. Twój czas wykonania może się różnić.)*

## Tło

Ten samouczek pokazuje, jak zbudować, wdrożyć i uruchomić `wzorzec Qiskit` do symulacji łańcucha Heisenberga i estymacji energii jego stanu podstawowego. Więcej informacji o `wzorcach Qiskit` i o tym, jak `Qiskit Serverless` może być używany do ich wdrażania w chmurze dla zarządzanego wykonywania, znajdziesz na naszej [stronie dokumentacji dotyczącej IBM Quantum&reg; Platform](/guides/serverless).

## Wymagania

Przed rozpoczęciem tego samouczka upewnij się, że masz zainstalowane następujące pakiety:

* Qiskit SDK w wersji 1.2 lub nowszej, z obsługą [wizualizacji](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime w wersji 0.28 lub nowszej (`pip install qiskit-ibm-runtime`)
* Qiskit Serverless (pip install qiskit_serverless)
* IBM Catalog (pip install qiskit-ibm-catalog)

## Konfiguracja

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from typing import Sequence


from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.base import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, Estimator

from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

In [2]:
def visualize_results(results):
    plt.plot(results["cost_history"], lw=2)
    plt.xlabel("Iteration")
    plt.ylabel("Energy")
    plt.show()


def build_callback(
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
    callback_dict: dict,
):
    def callback(current_vector):
        # Keep track of the number of iterations
        callback_dict["iters"] += 1
        # Set the prev_vector to the latest one
        callback_dict["prev_vector"] = current_vector
        # Compute the value of the cost function at the current vector
        current_cost = (
            estimator.run([(ansatz, hamiltonian, [current_vector])])
            .result()[0]
            .data.evs[0]
        )
        callback_dict["cost_history"].append(current_cost)
        # Print to screen on single line
        print(
            "Iters. done: {} [Current cost: {}]".format(
                callback_dict["iters"], current_cost
            ),
            end="\r",
            flush=True,
        )

    return callback

## Krok 1: Mapowanie klasycznych danych wejściowych na problem kwantowy
*   Wejście: Liczba spinów
*   Wyjście: Ansatz i Hamiltonian modelujące łańcuch Heisenberga

Skonstruuj ansatz i Hamiltonian modelujące 10-spinowy łańcuch Heisenberga. Na początku importujemy kilka ogólnych pakietów i tworzymy kilka funkcji pomocniczych.

In [3]:
num_spins = 10
ansatz = efficient_su2(num_qubits=num_spins, reps=3)

# Remember to insert your token in the QiskitRuntimeService constructor
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, min_num_qubits=num_spins, simulator=False
)

coupling = backend.target.build_coupling_map()
reduced_coupling = coupling.reduce(list(range(num_spins)))

edge_list = reduced_coupling.graph.edge_list()
ham_list = []

for edge in edge_list:
    ham_list.append(("ZZ", edge, 0.5))
    ham_list.append(("YY", edge, 0.5))
    ham_list.append(("XX", edge, 0.5))

for qubit in reduced_coupling.physical_qubits:
    ham_list.append(("Z", [qubit], np.random.random() * 2 - 1))

hamiltonian = SparsePauliOp.from_sparse_list(ham_list, num_qubits=num_spins)

ansatz.draw("mpl", style="iqp")

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif)

## Krok 2: Optymalizacja problemu pod kątem wykonania na sprzęcie kwantowym
*   Wejście: Abstrakcyjny Circuit, obserwowalny
*   Wyjście: Docelowy Circuit i obserwowalny, zoptymalizowane dla wybranego QPU

Użyj funkcji `generate_preset_pass_manager` z Qiskit, aby automatycznie wygenerować procedurę optymalizacji naszego Circuit względem wybranego QPU. Wybieramy `optimization_level=3`, co zapewnia najwyższy poziom optymalizacji spośród wstępnie zdefiniowanych menedżerów przejść. Dodajemy również przejścia harmonogramowania `ALAPScheduleAnalysis` i `PadDynamicalDecoupling`, aby tłumić błędy dekoherencji.

In [4]:
target = backend.target
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
pm.scheduling = PassManager(
    [
        ALAPScheduleAnalysis(durations=target.durations()),
        PadDynamicalDecoupling(
            durations=target.durations(),
            dd_sequence=[XGate(), XGate()],
            pulse_alignment=target.pulse_alignment,
        ),
    ]
)
ansatz_ibm = pm.run(ansatz)
observable_ibm = hamiltonian.apply_layout(ansatz_ibm.layout)
ansatz_ibm.draw("mpl", scale=0.6, style="iqp", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif)

## Krok 3: Wykonanie przy użyciu prymitywów Qiskit
*   Wejście: Docelowy Circuit i obserwowalny
*   Wyjście: Wyniki optymalizacji

Minimalizuj szacowaną energię stanu podstawowego systemu poprzez optymalizację parametrów Circuit. Użyj prymitywu `Estimator` z Qiskit Runtime do obliczania funkcji kosztu podczas optymalizacji.

W tej demonstracji uruchomimy kod na QPU, używając prymitywów `qiskit-ibm-runtime`. Aby uruchomić z prymitywami opartymi na wektorach stanów `qiskit`, zastąp blok kodu używający prymitywów Qiskit IBM Runtime skomentowanym blokiem.

In [ ]:
# SciPy minimizer routine
def cost_func(
    params: Sequence,
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
) -> float:
    """Ground state energy evaluation."""
    return (
        estimator.run([(ansatz, hamiltonian, [params])])
        .result()[0]
        .data.evs[0]
    )


num_params = ansatz_ibm.num_parameters
params = 2 * np.pi * np.random.random(num_params)

callback_dict = {
    "prev_vector": None,
    "iters": 0,
    "cost_history": [],
}

# Evaluate the problem on a QPU by using Qiskit IBM Runtime
with Session(backend=backend) as session:
    estimator = Estimator()
    callback = build_callback(
        ansatz_ibm, observable_ibm, estimator, callback_dict
    )
    res = minimize(
        cost_func,
        x0=params,
        args=(ansatz_ibm, observable_ibm, estimator),
        callback=callback,
        method="cobyla",
        options={"maxiter": 100},
    )

visualize_results(callback_dict)

## Krok 4: Przetwarzanie końcowe i zwrócenie wyniku w żądanym formacie klasycznym
*   Wejście: Estymacje energii stanu podstawowego podczas optymalizacji
*   Wyjście: Szacowana energia stanu podstawowego

In [ ]:
print(f'Estimated ground state energy: {res["fun"]}')

## Wdrożenie wzorca Qiskit w chmurze
Aby to zrobić, przenieś powyższy kod źródłowy do pliku `./source/heisenberg.py`, opakuj kod w skrypt przyjmujący dane wejściowe i zwracający końcowe rozwiązanie, a następnie prześlij go do zdalnego klastra za pomocą klasy `QiskitFunction` z `qiskit-ibm-catalog`. Wskazówki dotyczące określania zewnętrznych zależności, przekazywania argumentów wejściowych i nie tylko znajdziesz w [przewodnikach po Qiskit Serverless](/guides/serverless).

Wejściem do wzorca jest liczba spinów w łańcuchu. Wyjściem jest estymacja energii stanu podstawowego systemu.

In [ ]:
# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()
heisenberg_function = QiskitFunction(
    title="ibm_heisenberg",
    entrypoint="heisenberg.py",
    working_dir="./source/",
)
serverless.upload(heisenberg_function)

### Uruchamianie wzorca Qiskit jako usługi zarządzanej
Po przesłaniu wzorca do chmury możemy go łatwo uruchomić za pomocą klienta `QiskitServerless`.

In [ ]:
# Run the pattern on the remote cluster

ibm_heisenberg = serverless.load("ibm_heisenberg")
job = serverless.run(ibm_heisenberg)
solution = job.result()

print(solution)
print(job.logs())

## Ankieta dotycząca samouczka
Prosimy o wypełnienie krótkiej ankiety, aby podzielić się opinią na temat tego samouczka. Twoje spostrzeżenia pomogą nam ulepszyć nasze treści i doświadczenie użytkownika.

[Link do ankiety](https://your.feedback.ibm.com/jfe/form/SV_bfuBwfNeeFBxnim)

> **Note:** This survey is provided by IBM Quantum and relates to the original English content. To give feedback on doQumentation's website, translations, or code execution, please [open a GitHub issue](https://github.com/JanLahmann/doQumentation/issues).